# Evidence-Aware Multi-Agent RAG for Scientific Literature Review

**CSE427 Final Project — Group 10**

This faculty-ready notebook reproduces the project setup, validates the QASPER processing pipeline, runs a small real validation-only demonstration, and presents committed experimental results. Our contributions are paragraph-aware preprocessing, a unified paragraph/section/figure-table corpus, hybrid retrieval with evidence-aware reranking, and a traceable five-agent answer workflow.

## A. Project objective and contributions

The research objective is to test whether explicit evidence selection and bounded multi-agent critique improve the transparency and practical behavior of scientific question answering. Inference and gold evaluation are deliberately separate: the live demo reads question metadata only, while the result section loads previously committed aggregate validation results.

## B. System architecture and five agents

`Question → Query Agent → Retrieval Agent (BM25 + BGE dense + weighted RRF) → Evidence Agent (cross-encoder + fused scoring, E1–E5) → Answer Agent (Qwen) → Critic Agent → answer, citations, public trace`

The Query Agent normalizes the request; Retrieval combines paper-scoped lexical and dense candidates; Evidence reranks and labels five sources; Answer generates only from selected evidence; Critic performs deterministic citation/grounding checks plus one bounded review/revision.

## C. Environment and dependency setup

In [ ]:
# Execution controls: defaults never rerun the fixed 100-question generation experiment.
QUICK_DEMO = True
RUN_FULL_GENERATION = False
REBUILD_INDEX_IF_MISSING = True
DEMO_QUESTION_COUNT = 3
SEED = 427

REPOSITORY_URL = "https://github.com/raiyanhossainaraf-dagger/CSE427Labproject_S3_G10.git"
REPOSITORY_DIR = "CSE427Labproject_S3_G10"
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
GENERATION_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 256
GENERATION_BATCH_SIZE = 1

assert 3 <= DEMO_QUESTION_COUNT <= 5
if RUN_FULL_GENERATION:
    print("WARNING: RUN_FULL_GENERATION=True is optional, expensive, and reruns 100 deterministic generations.")

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    candidate = Path("/content") / REPOSITORY_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(candidate)], check=True)
    PROJECT_ROOT = candidate
    os.chdir(PROJECT_ROOT)
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "src").is_dir() and (p / "requirements-colab.txt").is_file()), None)
    if PROJECT_ROOT is None:
        raise RuntimeError("Run locally from this repository, or open the notebook in Colab.")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "-r", "requirements-llm.txt"], check=True)
print({"in_colab": IN_COLAB, "project_root": str(PROJECT_ROOT)})

In [ ]:
import gc, json, platform, random
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown
from src.utils import set_seed

set_seed(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
required = ["requirements-colab.txt", "requirements-llm.txt", "data/processed/questions.parquet",
            "outputs/tables/g5_answer_comparison.csv", "outputs/tables/g5_retrieval_comparison.csv"]
missing = [p for p in required if not (PROJECT_ROOT / p).is_file()]
if missing: raise FileNotFoundError(f"Required repository files missing: {missing}")
memory_gb = None
try:
    import psutil
    memory_gb = round(psutil.virtual_memory().total / 2**30, 1)
except ImportError: pass
env = {"python": platform.python_version(), "cuda_available": torch.cuda.is_available(),
       "cuda_version": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "memory_gb": memory_gb}
display(env)
if not torch.cuda.is_available():
    display(Markdown("**WARNING: GPU is unavailable. The exact requested models are retained—no substitute model is used—but the real demo will be slow on CPU.**"))

## D. QASPER dataset statistics and schema

QASPER contains questions grounded in NLP papers. The committed schema separates papers, sections, paragraphs, questions, answers, evidence, and figures/tables. The test data may exist as source data, but this notebook neither evaluates nor reports test-split performance.

In [ ]:
dataset_summary = json.loads((PROJECT_ROOT / "outputs/summaries/dataset_summary.json").read_text())
schema_summary = json.loads((PROJECT_ROOT / "outputs/summaries/qasper_schema_summary.json").read_text())
display(dataset_summary, schema_summary)

## E. Evidence mapping and validation

In [ ]:
mapping_summary = json.loads((PROJECT_ROOT / "outputs/summaries/evidence_mapping_summary.json").read_text())
assert mapping_summary["unmatched"] == 0 if "unmatched" in mapping_summary else mapping_summary["status_counts"]["unmatched"] == 0
display(mapping_summary)

## F. Paragraph-aware chunking

Chunks preserve paragraph identifiers and controlled overlap instead of treating papers as undifferentiated text. The committed summary below is descriptive; rebuilding the retrieval corpus uses the repository implementation in `src.retrieval_corpus`.

In [ ]:
chunking_summary = json.loads((PROJECT_ROOT / "outputs/summaries/chunking_summary.json").read_text())
display(chunking_summary)

## G. Unified paragraph/section/figure-table retrieval corpus

In [ ]:
from src.config import RETRIEVAL_ARTIFACT_VERSION
artifact_dir = PROJECT_ROOT / "data/processed" / RETRIEVAL_ARTIFACT_VERSION
corpus_path = artifact_dir / "corpus.parquet"
if not corpus_path.exists():
    subprocess.run([sys.executable, "scripts/build_retrieval_corpus.py"], cwd=PROJECT_ROOT, check=True)
corpus = pd.read_parquet(corpus_path)
display(corpus.groupby(["split", "source_type"]).size().rename("documents").to_frame())

## H. Dense index construction when ignored artifacts are absent

In [ ]:
dense_required = ["passage_embeddings.f32.npy", "index_flat_ip.faiss", "document_map.parquet", "dense_manifest.json"]
dense_missing = [name for name in dense_required if not (artifact_dir / name).exists()]
if dense_missing:
    if not REBUILD_INDEX_IF_MISSING:
        raise FileNotFoundError(f"Dense artifacts missing and rebuild disabled: {dense_missing}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    subprocess.run([sys.executable, "scripts/build_dense_index.py", "--device", device], cwd=PROJECT_ROOT, check=True)
display(json.loads((artifact_dir / "dense_manifest.json").read_text()))

## I. BM25, dense, hybrid RRF and evidence-aware reranking

The repository implements BM25 and BGE dense retrieval over the unified corpus, combines rankings with weighted reciprocal-rank fusion, then applies `cross-encoder/ms-marco-MiniLM-L6-v2` and the frozen evidence fusion rule. Missing requested models raise errors; no model is silently substituted.

In [ ]:
from src.bm25_retrieval import PaperScopedBM25Retriever
from src.embeddings import load_embedding_model
from src.hybrid_retrieval import HybridRetriever
from src.reranker import CrossEncoderReranker
print({"dense": DENSE_MODEL, "reranker": RERANKER_MODEL, "rrf": "src.hybrid_retrieval.weighted_rrf"})

## J. Query, Retrieval, Evidence, Answer and Critic Agents

In [ ]:
from src.query_agent import QueryAgent
from src.retrieval_agent import RetrievalAgent
from src.evidence_agent import EvidenceAgent
from src.answer_agent import AnswerAgent
from src.critic_agent import CriticAgent
from src.llm_backend import TransformersLLMBackend
from src.orchestrator import MultiAgentOrchestrator
print([QueryAgent.__name__, RetrievalAgent.__name__, EvidenceAgent.__name__, AnswerAgent.__name__, CriticAgent.__name__])

## K. Small real end-to-end validation demonstration

This inference cell reads only `questions.parquet`, whose schema contains question metadata and no gold answers or gold evidence. It uses 3–5 deterministically selected validation questions. CUDA uses FP16 for Qwen; generation is greedy (`do_sample=False`), `max_new_tokens=256`, batch size 1. CPU is allowed only with the same frozen Qwen model and may be slow.

In [ ]:
# Gold-free live inference boundary.
questions = pd.read_parquet(PROJECT_ROOT / "data/processed/questions.parquet")
for forbidden in ("answer", "answers", "evidence", "gold_answer", "gold_evidence"):
    assert forbidden not in questions.columns
demo_questions = questions.loc[questions.split.eq("validation"), ["question_id", "paper_id", "split", "question"]].sort_values(
    ["paper_id", "question_id"], kind="stable").head(DEMO_QUESTION_COUNT)
display(demo_questions)

In [ ]:
from scripts.run_multi_agent_demo import build_flow
from src.llm_backend import DEFAULT_MODEL_NAME

assert DEFAULT_MODEL_NAME == GENERATION_MODEL
demo_results = []
if QUICK_DEMO:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    backend = TransformersLLMBackend(model_name=GENERATION_MODEL, max_new_tokens=MAX_NEW_TOKENS, device=device)
    assert backend.generation_config == {"do_sample": False, "max_new_tokens": 256, "batch_size": 1}
    flow = build_flow(PROJECT_ROOT, backend, device=device, use_llm_critic=True)
    for row in demo_questions.itertuples(index=False):
        demo_results.append(flow.run(str(row.question_id), row.question, str(row.paper_id), "validation"))
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print("QUICK_DEMO=False: live inference skipped.")

## L. Questions, E1–E5 evidence, answers, citations and public traces

In [ ]:
for result in demo_results:
    display(Markdown(f"### {result.question}\n\n**Final answer:** {result.answer}\n\n**Citations:** " +
                     (", ".join(c["label"] for c in result.citations) or "None") +
                     f"\n\n**Public trace status:** `{result.final_status}`"))
    evidence_rows = [{"label": e["citation_label"], "source_type": e["source_type"],
                      "source_id": e["source_id"], "section": e["section_name"], "title": e["title"],
                      "evidence": e["evidence_text"]} for e in result.selected_evidence]
    display(pd.DataFrame(evidence_rows))
    display(pd.DataFrame(result.agent_traces)[["agent_name", "status", "output_count", "decisions"]])

## M. Saved G5 results (no 100-question rerun)

The fixed deterministic generation sample contains exactly 100 validation questions. The following cell loads committed aggregates and predictions; it does not invoke generation or load gold annotations.

In [ ]:
answer_results = pd.read_csv(PROJECT_ROOT / "outputs/tables/g5_answer_comparison.csv")
retrieval_results = pd.read_csv(PROJECT_ROOT / "outputs/tables/g5_retrieval_comparison.csv")
sample = pd.read_csv(PROJECT_ROOT / "outputs/tables/g5_validation_sample.csv")
assert len(sample) == 100 and sample.split.eq("validation").all()
display(answer_results, retrieval_results.query("k == 5"))

if RUN_FULL_GENERATION:
    raise RuntimeError("Expensive rerun is intentionally not automatic. Run `python scripts/run_final_experiments.py` only with explicit authorization and suitable GPU time.")

## N. Retrieval, reranking and Answer F1 comparison charts

Retrieval metrics cover **927 evidence-eligible validation questions**. Generation metrics cover a separate fixed deterministic **100-question validation sample**.

In [ ]:
from IPython.display import Image
for filename in ["retrieval_comparison.png", "answer_f1_comparison.png", "runtime_comparison.png", "answer_type_diagnostic.png"]:
    display(Image(filename=str(PROJECT_ROOT / "figures" / filename)))

## O. Paired statistical diagnostic and interpretation

Full minus dense Answer F1 is **−2.47 percentage points**; wins/ties/losses are **20/50/30**; the bootstrap 95% CI is **−8.41 to +3.49 points**; paired permutation **p = 0.419**. The interval crosses zero and the test is not significant.

> The full multi-agent system produced statistically comparable Answer F1 to the dense single-agent baseline on the fixed 100-question sample, while achieving higher evidence retrieval Recall@5, fewer insufficient responses, traceable evidence selection, and validated citation labels.

Citation-label validity verifies that emitted labels refer to selected E1–E5 records; it does **not** establish semantic correctness. No retrieval significance claim is made because no paired retrieval significance test is reported.

## P. Limitations, reproducibility and conclusion

- Full-validation retrieval and 100-question generation are different evaluation scopes.
- The fixed sample is not full-validation generation; Answer F1 is statistically comparable, not higher, for the full system.
- Citation-label validity is structural, not proof of factual support.
- QASPER is domain-specific, and CPU execution is slow; a Colab GPU is strongly recommended.
- Reproducibility uses seed 427, frozen model identifiers, greedy generation, committed aggregates, and rebuildable ignored indexes.

The project demonstrates stronger evidence retrieval and more traceable behavior without overstating generation quality. A fresh Colab clone installs repository dependencies, builds the ignored unified corpus/index if needed, runs only the small validation demo by default, and loads committed full-result tables.